# CS781 Capstone Phase 7: Prior Case Retrieval (PCR)
## Official Appendix Audit, Benchmark Analysis & Visualization Notebook
**Authors:** CS781 Capstone Research Group  
**Date:** November 2026  
**Lineage:** 100% Phase 1 Backbones (V1, V2, V3, V4, KD) — Zero External Pretrained Models  
**Primary Metrics:** Micro-F1@7, Micro-P@7, Micro-R@7, MRR, Recall@K

In [ ]:
import os
import sys
import json
import math
import zipfile
import numpy as np
import matplotlib.pyplot as plt

BASE_DIR = r'D:\Phase7_PCR_050_PLUS'
DATA_DIR = r'C:\Users\LENOVO\.gemini\antigravity\scratch\CS781_Capstone\Phase_7_PCR\data'

with open(os.path.join(DATA_DIR, 'Dev', 'dev.json'), 'r', encoding='utf-8') as f:
    dev_q = json.load(f)
with open(os.path.join(DATA_DIR, 'Dev', 'dev_candidates.json'), 'r', encoding='utf-8') as f:
    dev_c = json.load(f)
with open(os.path.join(DATA_DIR, 'Test', 'test.json'), 'r', encoding='utf-8') as f:
    test_q = json.load(f)
with open(os.path.join(DATA_DIR, 'Test', 'test_candidates.json'), 'r', encoding='utf-8') as f:
    test_c = json.load(f)

print(f'Loaded Dev: {len(dev_q)} queries, {len(dev_c)} candidates')
print(f'Loaded Test: {len(test_q)} queries, {len(test_c)} candidates')

### 1. Mathematical Upper Bound (Oracle Ceiling) Calculation
Because the competition strictly requires outputting 7 candidates per query, any query with $|G_q| < 7$ is guaranteed to produce $7 - |G_q|$ unavoidable False Positives.

In [ ]:
g_counts = [len(v['relevant_candidates']) for v in dev_q.values()]
oracle_max_tp = sum(min(c, 7) for c in g_counts)
total_golds = sum(g_counts)
total_preds = len(dev_q) * 7
unavoidable_fp = total_preds - oracle_max_tp
oracle_max_f1 = (2 * oracle_max_tp) / (total_golds + total_preds)

print(f'Total Gold Precedent Instances: {total_golds}')
print(f'Queries with < 7 Golds: {sum(1 for c in g_counts if c < 7)} / {len(dev_q)} ({sum(1 for c in g_counts if c < 7)/len(dev_q):.2%})')
print(f'Unavoidable False Positives: {unavoidable_fp} slots')
print(f'Oracle Theoretical Max True Positives: {oracle_max_tp} / {total_preds}')
print(f'Absolute Theoretical Oracle Micro-F1@7: {oracle_max_f1:.4f}')

### 2. Live Reproduction of Official Submission Leaderboard (Tracks E1 - E6)
Evaluating all 6 submissions using the exact official scoring metric.

In [ ]:
def evaluate_retrieval(predictions, ground_truth, k=7):
    total_tp = 0
    total_preds = 0
    total_golds = 0
    rec_at_k = {10: 0, 20: 0, 50: 0, 100: 0, 150: 0}
    mrr_sum = 0.0
    num_queries = len(ground_truth)
    
    for q_id, q_data in ground_truth.items():
        golds = set(q_data.get('relevant_candidates', []))
        total_golds += len(golds)
        preds = predictions.get(q_id, [])
        top_k = preds[:k]
        total_preds += len(top_k)
        hits = set(top_k) & golds
        total_tp += len(hits)
        first_hit = None
        for r, p in enumerate(preds, 1):
            if p in golds and first_hit is None:
                first_hit = r
        if first_hit:
            mrr_sum += 1.0 / first_hit
        for cut in rec_at_k:
            rec_at_k[cut] += len(set(preds[:cut]) & golds) / max(len(golds), 1)
            
    p = total_tp / max(total_preds, 1)
    r = total_tp / max(total_golds, 1)
    f1 = 2 * p * r / max(p + r, 1e-12)
    res = {
        'Micro_P@7': round(p, 4),
        'Micro_R@7': round(r, 4),
        'Micro_F1@7': round(f1, 4),
        'Total_TP': total_tp,
        'MRR': round(mrr_sum / max(num_queries, 1), 4)
    }
    for cut in rec_at_k:
        res[f'Recall@{cut}'] = round(rec_at_k[cut] / max(num_queries, 1), 4)
    return res

# Load saved dev predictions
track_files = {
    'E1: V1 LoRMA+ (MultiChunk)': os.path.join(BASE_DIR, 'predictions', 'E1_MultiChunk_dev_predictions.json'),
    'E2: V2 LayerWise': os.path.join(r'D:\Phase7_PCR', 'predictions', 'dev', 'E2_V2_LayerWise_dev_predictions.json'),
    'E3: V3 (IA)^3': os.path.join(r'D:\Phase7_PCR', 'predictions', 'dev', 'E3_V3_IA3_dev_predictions.json'),
    'E4: V4 Prefix (MultiChunk)': os.path.join(BASE_DIR, 'predictions', 'E4_MultiChunk_dev_predictions.json'),
    'E5: KD Student 47.5M': os.path.join(r'D:\Phase7_PCR', 'predictions', 'dev', 'E5_KD_Student_dev_predictions.json')
}

results = {}
for tname, tpath in track_files.items():
    if os.path.exists(tpath):
        with open(tpath, 'r', encoding='utf-8') as f:
            preds = json.load(f)
        results[tname] = evaluate_retrieval(preds, dev_q, k=7)
        print(f"{tname:28s} -> F1@7: {results[tname]['Micro_F1@7']} | TP: {results[tname]['Total_TP']} | MRR: {results[tname]['MRR']}")

### 3. Displaying Appendix Visualizations
Rendering the high-resolution publication figures generated from empirical evaluation logs.

In [ ]:
from IPython.display import Image, display

fig_names = [
    'fig1_track_leaderboard.png',
    'fig2_recall_at_k.png',
    'fig3_ceiling_error_decomposition.png',
    'fig4_ablation_progression.png',
    'fig5_e1_training_convergence.png',
    'fig6_citation_precision_comparison.png'
]

for fname in fig_names:
    fpath = os.path.join(BASE_DIR, 'figures', fname)
    if os.path.exists(fpath):
        print(f'=== Displaying {fname} ===')
        display(Image(filename=fpath))
